## HDFS DeepLog Anomaly Detection Experiment

This experiment focuses on training and evaluating the **DeepLog** sequence model for **anomaly detection** on the HDFS log dataset.

### I. Dataset and Source

* **Dataset:** HDFS (LogHub preprocessed)
* **Source Citation:** Wei Xu et al. (SOSP 2009), Jieming Zhu et al. (ISSRE 2023)

### II. Model: DeepLog (LSTM-based)

* **Architecture:** **LSTM-based sequence anomaly detector**
* **Hyperparameters:**
    * `embedding_dim`: 64
    * `hidden_dim`: 128
    * `num_layers`: 2
    * `dropout`: 0.3


### III. Task and Evaluation

* **Task:** **Next-Event Prediction** for Sequence-Based Anomaly Detection.
    * Anomalies detected using **Top-k Prediction** criteria.
    * `top_k`: 4
* **Evaluation Metrics:** Accuracy, Precision, Recall, F1 Score, and AUC.

### IV. Data Flow and Training

#### A. Preprocessing
* **Tokenization:** Log events are tokenized and mapped to **integer IDs** via a vocabulary.
* **Sequencing:** Log sequences are padded to the **95th percentile length** using a padding value of **0**.
* **Data Split:**
    * Train: 70%
    * Validation: 15%
    * Test: 15%

#### B. Training
* **Batch Size:** 64
* **Learning Rate:** 0.001
* **Epochs:** Up to 50, with **Early Stopping** (`patience=20`).
* **Device:** Auto-detected (mps/cpu).

### V. Artifacts

* **Model Checkpoint:** `../mdls/deeplog_checkpoint.pt`
* **Results:** `../results/hdfs_deeplog_results.json`


### 1. Setup and Imports

In [ ]:
# Core imports
import numpy as np
import torch
import sys
from pathlib import Path
# Add project root to Python path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import modules
from src.data.dataset import create_data_loaders
from src.models.deeplog import DeepLogModel
from src.engine.trainer import LogSeqTrainer
from src.utils.metrics import evaluate_model, print_metrics, save_experiment_results
from src.utils.data_loader import create_train_val_test_split, filter_normal_samples, load_loghub
from src.utils.visualizer import UniversalAnomalyVisualizer
from src.utils.seed import seed_everything

### 2. Load and Prepare Data

In [ ]:
seed_everything(42)  # For reproducibility

In [ ]:
# Define paths
DATA_DIR = '../data/hdfs/preprocessed'

In [ ]:
X, y, vocab = load_loghub(DATA_DIR)
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")
print(f"Events: {sorted(vocab.keys())[:10]}")  # See first 10

In [ ]:
#  Split data (70/15/15)
splits = create_train_val_test_split(X, y, train_ratio=0.7, val_ratio=0.15, random_state=42)
(X_train, y_train), (X_val, y_val), (X_test, y_test) = splits['train'], splits['val'], splits['test']
X_train, y_train = filter_normal_samples(X_train, y_train, verbose=True)

In [ ]:
#  Convert strings to integers
def convert_to_ids(sequences, vocab):
    return [[vocab[event] for event in seq] for seq in sequences]

X_train_ids = convert_to_ids(X_train, vocab)
X_val_ids = convert_to_ids(X_val, vocab)
X_test_ids = convert_to_ids(X_test, vocab)

In [ ]:
# pad sequences (integers, not strings!)
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = int(np.percentile([len(s) for s in X_train_ids], 95)) # todo try 64

X_train_padded = pad_sequences(X_train_ids, maxlen=max_len, padding='post', value=0)
X_val_padded = pad_sequences(X_val_ids, maxlen=max_len, padding='post', value=0)
X_test_padded = pad_sequences(X_test_ids, maxlen=max_len, padding='post', value=0)

print(f"Max length: {max_len}")
print(f"Train shape: {X_train_padded.shape}")

In [ ]:
# Get optimal dataloader kwargs
device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
loader_kwargs = LogSeqTrainer.get_optimal_dataloader_kwargs(device)
loader_kwargs

In [ ]:
# Create data loaders

batch_size = 64 # use higher based on device 
train_loader, val_loader, test_loader = create_data_loaders(
    X_train_padded, y_train,
    X_val_padded, y_val,
    X_test_padded, y_test,
    batch_size=batch_size,
    num_workers=loader_kwargs['num_workers']
)

### 3. Create Model

In [ ]:
#  Create model
model = DeepLogModel(
    vocab_size=vocab_size,
    embedding_dim=64,
    hidden_dim=128,
    num_layers=2,
    dropout=0.3 
)

### 4. Train

In [ ]:
# Train
print(f"Training on {device} device.")
learning_rate = 0.001
trainer = LogSeqTrainer(model, device=device, learning_rate=learning_rate)
patience = 10
history = trainer.fit(
    train_loader, val_loader,
    num_epochs=50,
    early_stopping_patience=patience,
    print_every=5
)

### 5. Evaluation

In [ ]:
# Evaluate
# Capture predictions, true labels, AND anomaly scores from the loader
top_k = 4                                                                       
predictions, true_labels, anomaly_scores = trainer.detect_anomalies(test_loader, top_k=top_k, return_scores=True)
metrics = evaluate_model(predictions, true_labels) 
print_metrics(metrics)

### 6. Visualizations

Generate comprehensive visualizations including:
1. Training history
2. Confusion matrix
3. Anomaly score distribution
4. ROC curve
5. Precision-Recall curve

In [ ]:
### 6. Visualizations
viz = UniversalAnomalyVisualizer(model_name="DeepLog", save_dir="../figures/deeplog")

# Plot 1: Training History (Loss curves)
viz.plot_training_history(history)

# Plot 2: Confusion Matrix
viz.plot_confusion_matrix(true_labels, predictions, title_suffix="(HDFS Test Set)")

# Plot 3: Anomaly Score Distribution
viz.plot_anomaly_score_distribution(true_labels, anomaly_scores, bins=50)

# Plot 4: ROC Curve
viz.plot_roc_curve(true_labels, anomaly_scores)

# Plot 5: Precision-Recall Curve
viz.plot_precision_recall_curve(true_labels, anomaly_scores)

print("\n✓ All visualizations generated and saved using UniversalAnomalyVisualizer")

In [ ]:
#  Save results
save_experiment_results(
    filepath="../results/hdfs_deeplog_results.json",
    dataset="HDFS",
    model_name="DeepLog",
    device=device,
    model=model,
    history=history,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
    max_len=max_len,
    metrics=metrics,
    batch_size=batch_size,
    learning_rate=learning_rate,
    patience=patience,
    top_k=top_k
)

In [ ]:
save_dir = Path('../mdls')
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / 'deeplog_checkpoint.pt'
trainer.save_model(str(save_path))